In [1]:
import numpy as np
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,Trainer,TrainingArguments,
    DataCollatorForTokenClassification)

from torch.utils.data import DataLoader, random_split, Dataset
from datasets import Dataset as DS
import json
import gc
from scipy.special import softmax
from spacy.lang.en import English

2024-04-23 20:42:35.560196: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-04-23 20:42:35.560323: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-04-23 20:42:35.692308: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


checkpoint = [
              "/kaggle/input/37vp4pjt",#
              "/kaggle/input/pii-models/piidd-org-sakura",#
              "/kaggle/input/pii-deberta-models/cola del piinguuino",#
    "/kaggle/input/pii-deberta-models/cabeza-de-piiranha",
    "/kaggle/input/pii-deberta-models/cuerpo del piinguuino",#
    "/kaggle/input/pii-deberta-models/cabeza-de-piiranha-mixtral-v1",#
    "/kaggle/input/pii-deberta-models/cola-de-piiranha-mixtral-v1",#
    "/kaggle/input/extra-small-3folds-1024-large/output/fold_1/checkpoint-16000"#
             ]

weights = [.1,.35,.25,.2,.15,.1,.1,.1]

checkpoint = [
              "/kaggle/input/37vp4pjt",
    "/kaggle/input/extra-small-3folds-1024-large/output/fold_1/checkpoint-16000",
    "/kaggle/input/pii-models/piidd-org-sakura",
    "/kaggle/input/pii-deberta-models/cabeza-del-piinguuino",
    "/kaggle/input/pii-deberta-models/cuerpo del piinguuino",
    "/kaggle/input/pii-deberta-models/cola del piinguuino"
             ]

weights = [.4,.07,.1,.2,.13,.1]

In [2]:
checkpoint = [
    '/kaggle/input/37vp4pjt',
    '/kaggle/input/pii-deberta-models/cuerpo-de-piiranha',
    '/kaggle/input/pii-deberta-models/cola del piinguuino' ,
    '/kaggle/input/pii-deberta-models/cabeza-del-piinguuino',
    '/kaggle/input/pii-deberta-models/cabeza-de-piiranha',
    '/kaggle/input/pii-deberta-models/cola-de-piiranha',
    '/kaggle/input/pii-models/piidd-org-sakura',
    '/kaggle/input/pii-deberta-models/cabeza-de-piiranha-persuade_v0'
    ]

weights = [1.,.15,.1,.5,.35,.1,.2,.1]

In [3]:
seq_len = 3700

In [4]:
def tokenize(example, tokenizer):
    
    tok = tokenizer(example["tokens"],
                    is_split_into_words = True,
            max_length = seq_len,
            truncation = True
            )
    word_ids = tok.word_ids()
    return {**tok,
            "word_ids":word_ids}

In [5]:
test_data = json.load(open("/kaggle/input/pii-detection-removal-from-educational-data/test.json"))

In [6]:
ds = DS.from_dict({
    "full_text": [x["full_text"] for x in test_data],
    "document": [x["document"] for x in test_data],
    "tokens": [x["tokens"] for x in test_data],
    "trailing_whitespace": [x["trailing_whitespace"] for x in test_data],
})

In [7]:
final_preds = None

In [8]:
%%time
for idx, chk in enumerate(checkpoint):
    pth = chk + '/config.json'
    ok = json.load(open(pth))
    label2id = ok["label2id"] 
    id2label = ok["id2label"]
    all_labels = len(id2label)
    
    tokenizer = AutoTokenizer.from_pretrained(chk)
    test_ds = ds.map(tokenize, fn_kwargs = {"tokenizer":tokenizer}, num_proc = 1)
    model = AutoModelForTokenClassification.from_pretrained(
        chk,
        ignore_mismatched_sizes=True,
        label2id=label2id,
        id2label=id2label,
        num_labels=all_labels)
    
    collator = DataCollatorForTokenClassification(tokenizer = tokenizer)
    
    args = TrainingArguments(".", per_device_eval_batch_size=1, report_to="none")
    trainer = Trainer(model=model, args=args, data_collator=collator, tokenizer=tokenizer,)
    torch.cuda.empty_cache()
    
    predsT = trainer.predict(test_ds)
    predsT = softmax(predsT.predictions, axis = -1)
    del tokenizer,model,collator,trainer,args
    gc.collect()
    if idx == 0:
        final_preds = predsT*weights[idx]
    else:
        final_preds += predsT*weights[idx]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

/opt/conda/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


CPU times: user 1min 12s, sys: 9.9 s, total: 1min 21s
Wall time: 2min 45s


In [9]:
final_preds = final_preds/sum(weights)

In [10]:
noO = final_preds[:,:,:12].argmax(-1)
Opreds = final_preds[:,:,12]
act_preds = final_preds.argmax(-1)
predsT_ = np.where(Opreds<.93, noO, act_preds)

In [11]:
predsT_ = predsT_[:,1:]

In [12]:
predT_ = np.array(predsT_)

In [13]:
%%time
tok_id = []
docu = []
pred_tags = []
token_ = []

for idx,i in enumerate(predsT_):
    ln = np.where(np.array(test_ds[idx]["word_ids"][1:])==None)[0][0]
    st=0
    
    while st<=ln:
    
        if i[st]<all_labels-1:
            docu.append(test_ds[idx]["document"])
            tok_id.append(test_ds[idx]["word_ids"][st+1])
            token_.append(test_data[idx]["tokens"][tok_id[-1]])#
            pred_tags.append(id2label[str(i[st])])
            
        if i[ln]<all_labels-1:
            docu.append(test_ds[idx]["document"])
            tok_id.append(test_ds[idx]["word_ids"][ln+1])
            token_.append(test_data[idx]["tokens"][tok_id[-1]])
            pred_tags.append(id2label[str(i[ln])])
            
        if st == ln:
            if i[st]<all_labels-1:
                docu.append(test_ds[idx]["document"])
                tok_id.append(test_ds[idx]["word_ids"][st+1])
                token_.append(test_data[idx]["tokens"][tok_id[-1]])
                pred_tags.append(id2label[str(i[st])])

        st+=1
        ln-=1
    
    
sub = pd.DataFrame()
sub["document"] = docu
sub["token"] = tok_id
sub["tokens"] = token_
sub["label"] = pred_tags

sub.drop_duplicates(subset = ["document", "token"], keep = 'first', inplace = True)

CPU times: user 335 ms, sys: 660 µs, total: 336 ms
Wall time: 342 ms


In [14]:
import re
phnum_ptrn = "(\([0-9]{3}\)[0-9]{3}\-[0-9]{4}([Xx]{1}([0-9]{3,5}))?)|([0-9]{3}\.[0-9]{3}\.[0-9]{4})"
idnum_ptrn = "[\d]{10}|[\d]{9}|((IV-)(\d){4})|([a-zA-Z]+_[\d]{10})|([a-zA-Z]+\|[\d]{8})|([a-zA-Z]{2}[\.]*[:]?([\d]{10}|[\d]{12}))"         
em_ptrn = "[\w\d.+-]+@[\w.-]+\.[\w.-]+"
st_ptrn1 = "[\d]+[a-zA-Z- \\n]+((Apt\. )|(HNo\.)|(#)|(hno\.))[\d]+([\s]*)[a-zA-z ]+([,]?)([\s]?)[a-zA-Z]{2}([\s]?)(\d{5})"
st_ptrn2 = "\d{1,4}[\s]?[\w]*[\s]?(ave|avenue|st|street|road|sq|square|highway|av|rd){1}[\.,]?[\s]?(((apt)|(hno)|(hno\.)|(apt\.))[\s\-]?\d+)?[\w\s,]*\d{5}[\w]?[\-,]?[\d]*[\w\s]*"
st_ptrn = f"({st_ptrn1})|({st_ptrn2})"

In [15]:
def PutIntoLists(idx, row, lab):
    tdoc.append(row["document"])
    tlab.append(lab)
    ttok.append(row["tokens"][idx])
    ttidx.append(idx)

In [16]:
def FindToken(tr,idx, match):
    """At dataset[`idx`][`full_text`] where `match` was found, 
    returns the `idx` with `token_index` which is used in dataset[`idx`][`tokens`][`token_index`]
    """
    text = ""
    mat = str.strip(match[0])
    tlen = len(mat)
    cur = ""
    ix = tlen-1
    while cur != " " and ix>=0:
        cur = mat[ix]
        text = cur + text
        ix-=1
    text = str.strip(text)
    
    tok_li = tr[idx]["tokens"]
    li_len = len(tok_li)-1
    st = 0
    while st<li_len:
        if re.search(text, tok_li[st]):
            return (idx, st)
        if re.search(text, tok_li[li_len]):
            return (idx, li_len)
        if st == li_len:
            if re.search(text, tok_li[li_len]):
                return (idx, li_len)
        st+=1
        li_len-=1 
    return (idx, None)

In [17]:
## id  :        027693
id_re = r"id[\s\.:]+ [\d]{5,9}" 

# Number : 047378465, without space at end it will also return all no or number. 12D 
# which is redudant as I am already using it in another case
num_re = r"((no)|(number))[\.]?[\s]*[:]?[\s]*[\d]{5,9} " 

# Code       :  06EYD876
code_re = r"code[\s]*[:]{1}[\s]*[a-z0-9]{4,12}"

url = "http[s]?://[a-z0-9\-\.\?=_/]{4,128}[\w.]+"
user_urls = "(homepage|contenthome|categor|listpost|channel|wp\-content|listmain|tagmain|tagsmain|searchprivacy|tagspost|tagpost|listindex|register|tate|postsabout|postabout|blogsearch|tagterms|exploreauthor|user|moore|mainlogin|blog|listfaq|tagprivacy|taghome|exploremain|appindex|exploreprivacy|explorelogin|tagssearch|owox|searchauthor|exploreabout|searchabout|tagsearch|taghomepage|blogindex|tagprivacy|postindex|postsindex|tagsauthor|linkedin|facebook|youtu\.be)+"
not_urls = r"(coursera|times|google|wiki|design|strateg|market|life|owox|inno|live|medic|uncat|voice|outc|whimsi|transl|(\.(ru|edu|gov)+)|lego)+"

In [18]:
tdoc = []
tlab = []
ttok = []
ttidx = []

In [19]:
for rid,row in enumerate(test_data):
    for idx, tok in enumerate(row["tokens"]):
        
        if re.match(em_ptrn, tok):
            PutIntoLists(idx, row, "B-EMAIL")
            
        elif re.match(idnum_ptrn, tok):
            PutIntoLists(idx, row, "B-ID_NUM")
            
        elif re.search(url, tok, re.IGNORECASE):
    
            if re.search(not_urls, tok, re.IGNORECASE):
                continue
            else:
                if re.search(user_urls, tok, re.IGNORECASE):
                    PutIntoLists(idx, row, 'B-URL_PERSONAL')
            
    if re.search(st_ptrn, row["full_text"]):
        matches = re.finditer(st_ptrn, row["full_text"], re.IGNORECASE)
        for m in matches:
            flag=0
            span = m.span()
            text = ""
            
            for tidx, tok_ in enumerate(row["tokens"]):
                if row["trailing_whitespace"][tidx]:
                    text += tok_ + " "
                else:
                    text += tok_
                
                if len(text)>= span[0]+1 and len(text)<= span[1]+1:
                    if flag==0:
                        PutIntoLists(tidx, row, "B-STREET_ADDRESS")
                        flag=1
                    else:
                        PutIntoLists(tidx, row, "I-STREET_ADDRESS")
                elif len(text)>span[1]+1:
                    break
    
    if re.search(phnum_ptrn, row["full_text"]):
        matches = re.finditer(phnum_ptrn, row["full_text"])
        for m in matches:
            flag=0
            span = m.span()
            text = ""
            
            for tidx, tok_ in enumerate(row["tokens"]):
                if row["trailing_whitespace"][tidx]:
                    text += tok_ + " "
                else:
                    text += tok_
                
                if len(text)>= span[0]+1 and len(text)<= span[1]+1:
                    if flag==0:
                        PutIntoLists(tidx, row, "B-PHONE_NUM")
                        flag=1
                    else:
                        PutIntoLists(tidx, row, "I-PHONE_NUM")
                elif len(text)>span[1]+1:
                    break
    
    if re.finditer(f"({id_re})|({num_re})|({code_re})", row["full_text"], re.IGNORECASE):
        for match in re.finditer(f"({id_re})|({num_re})|({code_re})", row["full_text"], re.IGNORECASE):
            lvl, tidx = FindToken(test_data,idx, match)
            if tidx is not None:
                PutIntoLists(tidx, row, "B-PHONE_NUM")

In [20]:
pdf = pd.DataFrame()
pdf["document"] = tdoc
pdf["token"] = ttok
pdf["token_idx"] = ttidx
pdf["label"] = tlab

In [21]:
%%time

for i in range(len(pdf)):
    if len(sub.query(f'document== {pdf["document"][i]} & token== {pdf["token_idx"][i]}'))>0:
        sub.loc[(sub.document==pdf.document[i]) & (sub.token==pdf.token_idx[i]), "label"]=pdf["label"][i]
    else:
        sub.loc[len(sub)]=[pdf.document[i], pdf.token_idx[i],pdf.token[i], pdf.label[i]]

CPU times: user 15 µs, sys: 2 µs, total: 17 µs
Wall time: 21.2 µs


In [22]:
sub = sub.sort_values(by = ["document", "token"])

In [23]:
sub.reset_index(drop = True, inplace = True)

In [24]:
cur = 0

while cur<len(sub):
            
    if sub["tokens"][cur]=="Dr" or sub["tokens"][cur]=="Dr." or sub["tokens"][cur]=="Mr." or sub["tokens"][cur]=="mr." or sub["tokens"][cur]=="Mr":
        sub = sub.drop(index = [cur], axis = 0)
        sub.reset_index(drop = True, inplace = True)
        cur-=1
        
    elif (sub["tokens"][cur] == "\n\n" or sub["tokens"][cur]  == "\n"or sub["tokens"][cur]==" ") and  sub["label"][cur][1:] !="-STREET_ADDRESS" :
        sub = sub.drop(index = [cur], axis = 0)
        sub.reset_index(drop = True, inplace = True)
        cur-=1
        
    elif sub["label"][cur][1:] == "-URL_PERSONAL":
        if re.search(not_urls, sub["tokens"][cur], re.IGNORECASE):
            sub = sub.drop(index = [cur], axis = 0)
            sub.reset_index(drop = True, inplace = True)
            cur-=1

        elif not re.search(url, sub["tokens"][cur], re.IGNORECASE):
            sub = sub.drop(index = [cur], axis = 0)
            sub.reset_index(drop = True, inplace = True)
            cur-=1
    
    elif sub["label"][cur][1:] == "-NAME_STUDENT":
        if re.match("[\d\.\-]", sub["tokens"][cur], re.IGNORECASE):
            sub = sub.drop(index = [cur], axis = 0)
            sub.reset_index(drop = True, inplace = True)
            cur-=1
            
        if len(sub["tokens"][cur])==1:
            sub = sub.drop(index = [cur], axis = 0)
            sub.reset_index(drop = True, inplace = True)
            cur-=1
    
    elif sub["label"][cur][1:] == "-ID_NUM":
        if len(sub["tokens"][cur])<=5:
            if re.match('[a-z]{1,5}$', sub["tokens"][cur], re.IGNORECASE):
                sub = sub.drop(index = [cur], axis = 0)
                sub.reset_index(drop = True, inplace = True)
                cur-=1
    
    elif sub["label"][cur][1:] == "-STREET_ADDRESS":
        if re.match("\d+:\d+", sub["tokens"][cur], re.IGNORECASE):
            sub = sub.drop(index = [cur], axis = 0)
            sub.reset_index(drop = True, inplace = True)
            cur-=1
    
    elif sub["label"][cur][1:] == "-PHONE_NUM" and (sub["tokens"][cur]=="(" or sub["tokens"][cur]==")"):
        try:
            print(sub["label"][cur+1][1:])
            if sub["label"][cur+1][1:] != "-PHONE_NUM" :
                sub = sub.drop(index = [cur], axis = 0)
                sub.reset_index(drop = True, inplace = True)
                cur-=1
        except:
            break
    
    cur+=1

In [25]:
sub.sort_values(by = ["document", "token"], inplace = True)
sub["row_id"] = [i for i in range(len(sub))]
sub.reset_index(drop= "index", inplace = True)

In [26]:
sub[[ "row_id","document", "token", "label",]].to_csv("submission.csv", index = False)

In [27]:
sub

,document,token,tokens,label,row_id
0,7,9,Nathalie,B-NAME_STUDENT,0
1,7,10,Sylla,I-NAME_STUDENT,1
2,7,482,Nathalie,B-NAME_STUDENT,2
3,7,483,Sylla,I-NAME_STUDENT,3
4,7,741,Nathalie,B-NAME_STUDENT,4
5,7,742,Sylla,I-NAME_STUDENT,5
6,10,0,Diego,B-NAME_STUDENT,6
7,10,1,Estrada,I-NAME_STUDENT,7
8,10,464,Diego,B-NAME_STUDENT,8
9,10,465,Estrada,I-NAME_STUDENT,9
